In [1]:
import torch
from mini_whisper import *
from mini_whisper.transformer.MHA_simple import SimpleTransformerBlock
from mini_whisper.transformer.MHA import TransformerBlock

In [2]:
SPLIT = "dev-clean"
DATA_DIR = "./data"
FOLDER_IN_ARCHIVE = "LibriSpeech"
BATCH_SIZE = 16
N_MELS = 80
D_MODEL = 128
N_HEADS = 8

print("=" * 60)
print("Mini-Whisper Training - Data Loading & Preprocessing")
print("=" * 60)


Mini-Whisper Training - Data Loading & Preprocessing


In [3]:
print(f"\nCreating DataLoader for: {DATA_DIR}")
print(f"Batch size: {BATCH_SIZE}")

dataloader = LibriSpeechAudioPreprocessingDataLoader(
    split=SPLIT,
    root_dir=DATA_DIR,
    folder_in_archive=FOLDER_IN_ARCHIVE,
    download_dataset=True,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    n_mel_bins=N_MELS
)

print(f"\nDataLoader created with {len(dataloader.dataset)} samples")
print(f"  Number of batches: {len(dataloader)}")



Creating DataLoader for: ./data
Batch size: 16

DataLoader created with 2703 samples
  Number of batches: 169


In [4]:
print(f"\nInitializing AudioEncoderStem (n_mels={N_MELS}, d_model={D_MODEL})")
stem = AudioEncoderStem(n_mels=N_MELS, d_model=D_MODEL)
stem.eval()
print("Encoder stem initialized")



Initializing AudioEncoderStem (n_mels=80, d_model=128)
Encoder stem initialized


In [5]:
print(f"\nProcessing first batch...")
batch = next(iter(dataloader))

log_mels = batch['log_mel']  # (B, n_mels, T)
transcripts = batch['transcript']
audio_paths = batch['audio_path']

print(f"\nBatch contents:")
print(f"  Log-mel shape: {log_mels.shape}")
print(f"  Number of transcripts: {len(transcripts)}")



Processing first batch...

Batch contents:
  Log-mel shape: torch.Size([16, 80, 3000])
  Number of transcripts: 16


In [6]:
print(f"\nFirst 3 samples in batch:")
for i in range(min(3, len(transcripts))):
    print(f"  {i+1}. {audio_paths[i]}")
    print(f"     Transcript: {transcripts[i][:60]}...")



First 3 samples in batch:
  1. 2086/149220/2086-149220-0012.flac
     Transcript: THE CHICKEN CREPT THROUGH THE PALES OF THE COOP AND RAN WITH...
  2. 2035/147961/2035-147961-0030.flac
     Transcript: PAVEL KNOCKED HIM OVER THE SIDE OF THE SLEDGE AND THREW THE ...
  3. 2277/149896/2277-149896-0008.flac
     Transcript: FOR SOME REASON HE FELT AS IF SOMETHING MIGHT COME THAT WAY ...


In [7]:
# Let's first start just with the stem
with torch.no_grad():
    batch_features = stem(log_mels)

print(f"\nEncoder output shape: {batch_features.shape}")



Encoder output shape: torch.Size([16, 1500, 128])


In [8]:
# Now let us add a layer of the encoder self-attention transformer block
print(f"\nInitializing AudioEncoderLayer (d_model={D_MODEL})")
encoder_layer = SimpleTransformerBlock(d_model=D_MODEL, n_head=N_HEADS)
with torch.no_grad():
    x = stem(log_mels)
    z = encoder_layer(x)
print(f"Output shape after encoder layer: {batch_features.shape}")



Initializing AudioEncoderLayer (d_model=128)
Output shape after encoder layer: torch.Size([16, 1500, 128])


In [9]:
# Okay, time for a encoder+decoder pair
print(f"\nInitializing AudioEncoderLayer (d_model={D_MODEL})")
encoder_layer = SimpleTransformerBlock(d_model=D_MODEL, n_head=N_HEADS)
decoder_layer = TransformerBlock(d_model=D_MODEL, n_head=N_HEADS, cross_attention=True)
with torch.no_grad():
    x = stem(log_mels)
    z = encoder_layer(x)
    y = decoder_layer(z, z)  # Using encoder output as both input and context for testing
print(f"Output shape after encoder+decoder layer: {y.shape}")


Initializing AudioEncoderLayer (d_model=128)
Output shape after encoder+decoder layer: torch.Size([16, 1500, 128])


In [10]:
# And now to decode the results
